# Trajectory Gradient Renderer (Latest Record)

Standalone notebook to load the latest saved run (`results.pkl` + `config.yaml`) and render:
- static gradient trajectory tracks (`star`, `planet`, `star+planet`)
- animated gradient evolution GIFs (`star`, `planet`, `star+planet`)

The configuration cell is intentionally the second code cell (right after imports).

Default output path: `branches/trajectory_audit_lab/audits/generated/<run_tag>/`


In [ ]:
from __future__ import annotations

from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

from slingshot.config import load_config
from slingshot.narrowed_baselines import compute_narrowed_baselines
from slingshot.plotting_twobody import plot_trajectory_tracks


In [ ]:
# Configuration (edit this cell)
RUN_DIR = None  # e.g. 'results/results_Kepler-432_20260228_194343'; None => auto-latest
RESULTS_ROOT = Path('results')
OUTPUT_ROOT = Path('branches/trajectory_audit_lab/audits/generated')
OUTPUT_TAG = None  # None => use run directory name

RENDER_MODE = 'both'  # 'static' | 'video' | 'both'

# Static rendering
STATIC_GRADIENT_MODE = 'legacy'  # legacy | line_overlay | hexbin | kde
STATIC_DPI = 180

# Video rendering
VIDEO_FPS = 10
VIDEO_DPI = 180
VIDEO_FRAMES = 60
VIDEO_HEXBIN_GRIDSIZE = 160
VIDEO_KDE_SIGMA_BINS = 2.0

# Shared trajectory controls
NUM_B = 160
NUM_ANGLES = 140
NUM_POINTS = 260
PADDING_FRAC = 0.20
MAX_OVERLAY_TRACKS = 260
OVERLAY_LINES = True
OVERLAY_LINE_COUNT = 110
CONFIDENCE_MIN_COUNT = 2
FIXED_ENERGY_RANGE = None  # e.g. (-5.0, 60.0)

# Standardized landscape sizing for report viewing
LANDSCAPE_FIGSIZE = (14, 8)


In [ ]:
def find_latest_run_dir(results_root: Path) -> Path:
    if not results_root.exists():
        raise FileNotFoundError(f"Results root does not exist: {results_root}")

    candidates = [
        p for p in results_root.iterdir()
        if p.is_dir() and (p / 'results.pkl').exists() and (p / 'config.yaml').exists()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"No run directories with results.pkl + config.yaml under: {results_root}"
        )
    return max(candidates, key=lambda p: p.stat().st_mtime)


run_dir = Path(RUN_DIR) if RUN_DIR else find_latest_run_dir(RESULTS_ROOT)
if not run_dir.exists():
    raise FileNotFoundError(f"Run directory not found: {run_dir}")

cfg = load_config(str(run_dir / 'config.yaml'))
with open(run_dir / 'results.pkl', 'rb') as f:
    record = pickle.load(f)

mc = record.get('mc')
rerun = record.get('rerun', {}) if isinstance(record, dict) else {}
sols_best = record.get('sols_best', []) or rerun.get('sols_best', []) or []
analyses_best = record.get('analyses_best', []) or rerun.get('analyses_best', []) or []
valid_analyses = [a for a in analyses_best if a is not None]
if not valid_analyses:
    raise RuntimeError('No valid analyses found in results.pkl; cannot build narrowed baselines.')

tb_cfg = cfg.two_body
narrowed = compute_narrowed_baselines(
    analyses_top=valid_analyses,
    cfg=cfg,
    padding_factor=tb_cfg.padding_factor if tb_cfg else 1.5,
    num_v=tb_cfg.num_v if tb_cfg else 20,
    num_b=tb_cfg.num_b_narrow if tb_cfg else 100,
    num_angles=tb_cfg.num_angles_narrow if tb_cfg else 100,
    verbose=False,
)

output_tag = OUTPUT_TAG.strip() if OUTPUT_TAG else run_dir.name
output_dir = OUTPUT_ROOT / output_tag
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Run directory: {run_dir}')
print(f'Output directory: {output_dir}')
print(f'MC success sample count: {len(mc.get("idx_success", [])) if isinstance(mc, dict) else "n/a"}')
print(f'Re-run solutions: {len(sols_best)} | valid analyses: {len(valid_analyses)}')


In [ ]:
def _save_static_combined(output_dir: Path, dpi: int = 180) -> Path:
    star_path = output_dir / 'trajectory_tracks_star.png'
    planet_path = output_dir / 'trajectory_tracks_planet.png'
    if not star_path.exists() or not planet_path.exists():
        missing = [str(p) for p in [star_path, planet_path] if not p.exists()]
        raise FileNotFoundError(f'Missing static input image(s): {missing}')

    img_star = plt.imread(star_path)
    img_planet = plt.imread(planet_path)

    fig, axes = plt.subplots(1, 2, figsize=LANDSCAPE_FIGSIZE, dpi=dpi, constrained_layout=True)
    axes[0].imshow(img_star)
    axes[0].set_title('Star Gradient Trajectory Tracks', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_planet)
    axes[1].set_title('Planet Gradient Trajectory Tracks', fontweight='bold')
    axes[1].axis('off')

    out_path = output_dir / 'trajectory_tracks_star_planet_combined.png'
    fig.savefig(out_path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    return out_path


def render_static_tracks() -> list[Path]:
    figs = plot_trajectory_tracks(
        narrowed=narrowed,
        sols_best=sols_best,
        analyses_best=analyses_best,
        cfg=cfg,
        num_b=NUM_B,
        num_angles=NUM_ANGLES,
        num_points=NUM_POINTS,
        padding_frac=PADDING_FRAC,
        max_overlay_tracks=MAX_OVERLAY_TRACKS,
        overlay_lines=OVERLAY_LINES,
        overlay_line_count=OVERLAY_LINE_COUNT,
        gradient_mode=STATIC_GRADIENT_MODE,
        confidence_min_count=CONFIDENCE_MIN_COUNT,
        fixed_energy_range=FIXED_ENERGY_RANGE,
        hexbin_gridsize=VIDEO_HEXBIN_GRIDSIZE,
        kde_sigma_bins=VIDEO_KDE_SIGMA_BINS,
        time_frames=VIDEO_FRAMES,
        export_phase_data=True,
        export_time_data=False,
        save_dir=str(output_dir),
        dpi=STATIC_DPI,
    )
    for fig in figs:
        plt.close(fig)

    generated = [
        output_dir / 'trajectory_tracks_star.png',
        output_dir / 'trajectory_tracks_planet.png',
        _save_static_combined(output_dir, dpi=STATIC_DPI),
    ]
    for p in generated:
        print(f'Saved static: {p}')
    return generated


In [ ]:
def _load_time_npz(npz_path: Path) -> dict:
    if not npz_path.exists():
        raise FileNotFoundError(f'Missing NPZ time data: {npz_path}')
    d = np.load(npz_path)
    return {
        'energy_frames': d['energy_frames'],
        'count_frames': d['count_frames'],
        'x_edges_km': d['x_edges_km'],
        'y_edges_km': d['y_edges_km'],
        'energy_vmin': float(d['energy_vmin']),
        'energy_vmax': float(d['energy_vmax']),
        'body': str(d['body']) if 'body' in d else 'unknown',
    }


def _make_single_body_gif(npz_path: Path, out_path: Path, title: str, fps: int, dpi: int) -> Path:
    data = _load_time_npz(npz_path)
    ef = data['energy_frames']
    x_edges = data['x_edges_km'] / 1e6
    y_edges = data['y_edges_km'] / 1e6
    n_frames = int(ef.shape[0])

    fig, ax = plt.subplots(figsize=LANDSCAPE_FIGSIZE, dpi=dpi)
    extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
    im = ax.imshow(
        ef[0].T,
        origin='lower',
        extent=extent,
        cmap='viridis',
        vmin=data['energy_vmin'],
        vmax=data['energy_vmax'],
        aspect='equal',
    )
    cb = fig.colorbar(im, ax=ax, pad=0.02)
    cb.set_label('Scattering energy 0.5|?V|? (km?/s?)')

    frame_text = ax.text(
        0.02,
        0.98,
        '',
        transform=ax.transAxes,
        va='top',
        ha='left',
        color='white',
        fontsize=11,
        bbox={'facecolor': 'black', 'alpha': 0.35, 'pad': 3, 'edgecolor': 'none'},
    )
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('X relative (10^6 km)')
    ax.set_ylabel('Y relative (10^6 km)')

    def _update(i: int):
        im.set_data(ef[i].T)
        frame_text.set_text(f'Frame {i + 1}/{n_frames}')
        return im, frame_text

    ani = FuncAnimation(fig, _update, frames=n_frames, interval=1000.0 / max(fps, 1), blit=False)
    ani.save(str(out_path), writer=PillowWriter(fps=fps), dpi=dpi)
    plt.close(fig)
    return out_path


def _make_combined_gif(star_npz: Path, planet_npz: Path, out_path: Path, fps: int, dpi: int) -> Path:
    ds = _load_time_npz(star_npz)
    dp = _load_time_npz(planet_npz)

    ef_s = ds['energy_frames']
    ef_p = dp['energy_frames']
    n_frames = int(min(ef_s.shape[0], ef_p.shape[0]))

    extent_s = [ds['x_edges_km'][0] / 1e6, ds['x_edges_km'][-1] / 1e6, ds['y_edges_km'][0] / 1e6, ds['y_edges_km'][-1] / 1e6]
    extent_p = [dp['x_edges_km'][0] / 1e6, dp['x_edges_km'][-1] / 1e6, dp['y_edges_km'][0] / 1e6, dp['y_edges_km'][-1] / 1e6]

    fig, axes = plt.subplots(1, 2, figsize=LANDSCAPE_FIGSIZE, dpi=dpi, constrained_layout=True)

    im_s = axes[0].imshow(
        ef_s[0].T,
        origin='lower',
        extent=extent_s,
        cmap='viridis',
        vmin=ds['energy_vmin'],
        vmax=ds['energy_vmax'],
        aspect='equal',
    )
    axes[0].set_title('Star Gradient Evolution', fontweight='bold')
    axes[0].set_xlabel('X relative (10^6 km)')
    axes[0].set_ylabel('Y relative (10^6 km)')
    cb_s = fig.colorbar(im_s, ax=axes[0], pad=0.01)
    cb_s.set_label('0.5|?V|?')

    im_p = axes[1].imshow(
        ef_p[0].T,
        origin='lower',
        extent=extent_p,
        cmap='viridis',
        vmin=dp['energy_vmin'],
        vmax=dp['energy_vmax'],
        aspect='equal',
    )
    axes[1].set_title('Planet Gradient Evolution', fontweight='bold')
    axes[1].set_xlabel('X relative (10^6 km)')
    axes[1].set_ylabel('Y relative (10^6 km)')
    cb_p = fig.colorbar(im_p, ax=axes[1], pad=0.01)
    cb_p.set_label('0.5|?V|?')

    frame_text = fig.text(0.5, 0.02, '', ha='center', va='center', fontsize=11)

    def _update(i: int):
        im_s.set_data(ef_s[i].T)
        im_p.set_data(ef_p[i].T)
        frame_text.set_text(f'Star + Planet Gradient Evolution | Frame {i + 1}/{n_frames}')
        return im_s, im_p, frame_text

    ani = FuncAnimation(fig, _update, frames=n_frames, interval=1000.0 / max(fps, 1), blit=False)
    ani.save(str(out_path), writer=PillowWriter(fps=fps), dpi=dpi)
    plt.close(fig)
    return out_path


def render_video_tracks() -> list[Path]:
    figs = plot_trajectory_tracks(
        narrowed=narrowed,
        sols_best=sols_best,
        analyses_best=analyses_best,
        cfg=cfg,
        num_b=NUM_B,
        num_angles=NUM_ANGLES,
        num_points=NUM_POINTS,
        padding_frac=PADDING_FRAC,
        max_overlay_tracks=MAX_OVERLAY_TRACKS,
        overlay_lines=OVERLAY_LINES,
        overlay_line_count=OVERLAY_LINE_COUNT,
        gradient_mode='time_video',
        confidence_min_count=CONFIDENCE_MIN_COUNT,
        fixed_energy_range=FIXED_ENERGY_RANGE,
        hexbin_gridsize=VIDEO_HEXBIN_GRIDSIZE,
        kde_sigma_bins=VIDEO_KDE_SIGMA_BINS,
        time_frames=VIDEO_FRAMES,
        export_phase_data=True,
        export_time_data=True,
        save_dir=str(output_dir),
        dpi=VIDEO_DPI,
    )
    for fig in figs:
        plt.close(fig)

    star_npz = output_dir / 'trajectory_time_data_star.npz'
    planet_npz = output_dir / 'trajectory_time_data_planet.npz'

    out_star = _make_single_body_gif(
        star_npz,
        output_dir / 'trajectory_gradient_evolution_star.gif',
        title='Trajectory Gradient Evolution (Star)',
        fps=VIDEO_FPS,
        dpi=VIDEO_DPI,
    )
    out_planet = _make_single_body_gif(
        planet_npz,
        output_dir / 'trajectory_gradient_evolution_planet.gif',
        title='Trajectory Gradient Evolution (Planet)',
        fps=VIDEO_FPS,
        dpi=VIDEO_DPI,
    )
    out_combined = _make_combined_gif(
        star_npz,
        planet_npz,
        output_dir / 'trajectory_gradient_evolution_star_planet.gif',
        fps=VIDEO_FPS,
        dpi=VIDEO_DPI,
    )

    generated = [out_star, out_planet, out_combined]
    for p in generated:
        print(f'Saved video: {p}')
    return generated


In [ ]:
generated_files: list[Path] = []

mode = str(RENDER_MODE).strip().lower()
if mode not in {'static', 'video', 'both'}:
    raise ValueError(f"RENDER_MODE must be one of static|video|both, got: {RENDER_MODE}")

if mode in {'static', 'both'}:
    generated_files.extend(render_static_tracks())

if mode in {'video', 'both'}:
    generated_files.extend(render_video_tracks())

print(f'Generated files: {len(generated_files)}')
for p in generated_files:
    print(f' - {p.name}')


In [ ]:
for p in generated_files:
    print(p.name)
    if p.suffix.lower() in {'.png', '.gif'} and p.exists():
        display(Image(filename=str(p)))
